# Script to run DINO on CT-images

In [7]:
# activated dependecies
import torch
import os
import torchvision
import pandas as pd
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from torchvision.transforms import v2
from transformers.image_utils import load_image
import torch.nn.functional as F
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [5]:
# set reference to DINO repo and weights
REPO_DIR = "dinov3"
CHECKPOINT_PATH = "weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"

# initiate pretrained DINOv3 ViT model 
dinov3_vits16 = torch.hub.load(REPO_DIR, 'dinov3_vits16', source='local', weights=CHECKPOINT_PATH)

## Functions to slice CT

In [ ]:
def resize_array(array, current_spacing, target_spacing):
    """
    Resize the array to match the target spacing.

    Args:
    array (torch.Tensor): Input array to be resized.
    current_spacing (tuple): Current voxel spacing (z_spacing, xy_spacing, xy_spacing).
    target_spacing (tuple): Target voxel spacing (target_z_spacing, target_x_spacing, target_y_spacing).

    Returns:
    np.ndarray: Resized array.
    """
    # Calculate new dimensions
    original_shape = array.shape[2:]
    scaling_factors = [
        current_spacing[i] / target_spacing[i] for i in range(len(original_shape))
    ]
    new_shape = [
        int(original_shape[i] * scaling_factors[i]) for i in range(len(original_shape))
    ]
    # Resize the array
    resized_array = F.interpolate(array, size=new_shape, mode='trilinear', align_corners=False).cpu().numpy()
    return resized_array

def nii_img_to_tensor(path):
    nii_img = nib.load(str(path))
    img_data = nii_img.get_fdata()

    slope = 1.0
    intercept = 0.0
    pixdim = nii_img.header.get_zooms()  # (x, y, z)
    xy_spacing = float(pixdim[0])
    z_spacing = float(pixdim[2])
    # print(f"Original spacing: x={xy_spacing}, y={xy_spacing}, z={z_spacing}")
    # Define the target spacing values
    target_x_spacing = 0.75
    target_y_spacing = 0.75
    target_z_spacing = 1.5
    # print(f"Target spacing: x={target_x_spacing}, y={target_y_spacing}, z={target_z_spacing}")

    current = (z_spacing, xy_spacing, xy_spacing)
    target = (target_z_spacing, target_x_spacing, target_y_spacing)

    img_data = slope * img_data + intercept
    hu_min, hu_max = -1000, 1000
    img_data = np.clip(img_data, hu_min, hu_max)

    img_data = img_data.transpose(2, 0, 1)

    tensor = torch.tensor(img_data)
    tensor = tensor.unsqueeze(0).unsqueeze(0)

    img_data = resize_array(tensor, current, target)
    img_data = img_data[0][0]
    img_data= np.transpose(img_data, (1, 2, 0))

    img_data = (((img_data ) / 1000)).astype(np.float32)
    slices=[]

    tensor = torch.tensor(img_data)
    # Get the dimensions of the input tensor
    target_shape = (480,480,240)

    # Extract dimensions
    h, w, d = tensor.shape
    # print(f"Original shape: h={h}, w={w}, d={d}")
    # print(f"Target shape: h={target_shape[0]}, w={target_shape[1]}, d={target_shape[2]}")

    # Calculate cropping/padding values for height, width, and depth
    dh, dw, dd = target_shape
    h_start = max((h - dh) // 2, 0)
    h_end = min(h_start + dh, h)
    w_start = max((w - dw) // 2, 0)
    w_end = min(w_start + dw, w)
    d_start = max((d - dd) // 2, 0)
    d_end = min(d_start + dd, d)

    # Crop or pad the tensor
    tensor = tensor[h_start:h_end, w_start:w_end, d_start:d_end]

    pad_h_before = (dh - tensor.size(0)) // 2
    pad_h_after = dh - tensor.size(0) - pad_h_before

    pad_w_before = (dw - tensor.size(1)) // 2
    pad_w_after = dw - tensor.size(1) - pad_w_before

    pad_d_before = (dd - tensor.size(2)) // 2
    pad_d_after = dd - tensor.size(2) - pad_d_before

    tensor = torch.nn.functional.pad(tensor, (pad_d_before, pad_d_after, pad_w_before, pad_w_after, pad_h_before, pad_h_after), value=-1)

    tensor = tensor.permute(2, 0, 1)

    tensor = tensor.unsqueeze(0)
    if tensor.ndim == 4:
        tensor = tensor.unsqueeze(1)

    return tensor

def plot_ct_slice(ct_tensor: torch.Tensor, slice_index: int):
    if ct_tensor.ndim != 5 or ct_tensor.shape[0] != 1 or ct_tensor.shape[1] != 1:
        raise ValueError(f"Expected shape [1, 1, D, H, W], got {tuple(ct_tensor.shape)}")
    
    depth = ct_tensor.shape[2]
    if not (0 <= slice_index < depth):
        raise ValueError(f"slice_index must be between 0 and {depth-1}, got {slice_index}")
    
    slice_2d = ct_tensor[0, 0, slice_index, :, :].cpu().numpy()
    
    plt.figure(figsize=(6, 6))
    plt.imshow(slice_2d, cmap='gray')
    plt.title(f"CT Slice {slice_index}/{depth-1}")
    plt.axis('off')
    plt.show()

def slice_ct(ct_tensor: torch.Tensor, slice_index: int):
    if ct_tensor.ndim != 5 or ct_tensor.shape[0] != 1 or ct_tensor.shape[1] != 1:
        raise ValueError(f"Expected shape [1, 1, D, H, W], got {tuple(ct_tensor.shape)}")
    
    depth = ct_tensor.shape[2]
    if not (0 <= slice_index < depth):
        raise ValueError(f"slice_index must be between 0 and {depth-1}, got {slice_index}")
    
    return ct_tensor[0, 0, slice_index, :, :].cpu().numpy()

## Functions to run Dino on a whole set of CTs

In [ ]:
def make_transform(resize_size: int = 256):
    to_tensor = v2.ToImage()
    resize = v2.Resize((resize_size, resize_size), antialias=True)
    to_float = v2.ToDtype(torch.float32, scale=True)
    normalize = v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return v2.Compose([to_tensor, resize, to_float, normalize])

def run_dino_on_single_img(image):

    # apllying transformation function (make tensor --> resize --> make float --> normalize)
    img_latents = make_transform()(image).unsqueeze(0)

    with torch.inference_mode():
        feats = dinov3_vits16.forward_features(img_latents) 

    if isinstance(feats, dict):
        pooled_output = feats["x_norm_clstoken"]
        patch_tokens = feats["x_norm_patchtokens"]
    else:
        # fallback if forward_features() not implemented
        tokens = model(img_latents) 
        pooled_output = tokens[:,0]          
        patch_tokens = tokens[:,1+model.num_register_tokens:,:]

    latents = [pooled_output, patch_tokens]
    return latents

def get_all_ct_latents(ct_image):
    # initiate lists
    global_latents_list = []
    local_latents_list = []

    # run dino on all slices
    for slice_index in range(ct_image.shape[2]):

        # extract slice
        ct_slice = slice_ct(ct_image, slice_index=slice_index)

        # create latents
        global_latents, local_latents = run_dino_on_single_img(ct_slice)
        
        # append latents of each step to one list
        global_latents_list.append(global_latents)
        local_latents_list.append(local_latents)

    # stack python list to a one tensor
    global_latents_stacked = torch.stack(global_latents_list)
    local_latents_stacked = torch.stack(local_latents_list)

    return [global_latents_stacked, local_latents_stacked]

def aggregate_ct_latents_to_one_dimension(latents, method="mean") -> torch.Tensor:
    
    # check whther input Tensor has the right dimensions
    if latents.ndim != 3:
        raise ValueError(f"Expected tensor of shape (n, 1, d), got {latents.shape}")

    # call the method which was chosen
    fn = METHODS.get(method)
    if fn is None:
        raise ValueError(f"Unknown method: {method}")
    return fn(latents.squeeze(1), dim=0).unsqueeze(0)


def gaussian_weights(n: int, center: float | None = None, sigma: float | None = None, dtype=None) -> torch.Tensor:

    # default values for mü and sigma for weights
    if center is None:
        center = (n - 1) / 2
    if sigma is None:
        sigma = n / 6.0

    # create gaussian weights for one dimension
    i = torch.arange(n, dtype=dtype)
    w = torch.exp(-0.5 * ((i - center) / sigma) ** 2)
    w = w / w.sum()
    return w

def agg_gaussian(x, **kwargs):

    # gaussian weights along first dimension
    n = x.size(0)
    w = gaussian_weights(n, dtype=x.dtype, **kwargs)
    return (x * w.unsqueeze(1)).sum(dim=0)

METHODS = {
    "mean": torch.mean,
    "sum": torch.sum,
    "gaussian": agg_gaussian,
}


def get_latents_full_data(num_files: int, base_path: str = "../data/rip_frac/ribfrac-val-images", file_ext: str = ".nii.gz", method = "mean") -> torch.Tensor:
    
    # get sorted list of all CT files in the directory
    all_files = sorted(
        [f for f in os.listdir(base_path) if f.endswith(file_ext)]
    )

    # make sure selected number of files is possible
    if num_files > len(all_files):
        num_files = len(all_files)
        print(f"⚠️ Only {num_files} files available; adjusted range.")


    all_global_latents_list = []

    # iterate through first `num_files` files
    for i, filename in enumerate(all_files[:num_files], start=1):
        path = os.path.join(base_path, filename)
        print(f"Processing file {i}/{num_files}: {filename}")

        # load CT volume
        ct_img = nii_img_to_tensor(path)

        # extract latents and average over slices
        latents_single_ct = get_all_ct_latents(ct_img)[0]

        agg_latents_single_ct = aggregate_ct_latents_to_one_dimension(latents_single_ct, method = method)

        all_global_latents_list.append(agg_latents_single_ct)

    # stack into single tensor
    all_latents = torch.stack(all_global_latents_list)

    print(f"✅ Finished: shape {all_latents.shape}")
    return all_latents